In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip uninstall -y datasets

Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1


In [3]:
!pip install datasets==2.17

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully uninstalled fsspec-2025.10.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Uninstalling dill-0.4.0:
      Successfully uninstalled dill-0.4.0
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.18
    Uninstalling multiprocess-0.70.18:
      Successfully uninstalled multiprocess-0.70.18
ERROR: pip's dependency resolver does 

In [4]:
import json
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer
from transformers import DistilBertForTokenClassification
from sklearn.metrics import accuracy_score, classification_report
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

2025-12-05 12:04:36.187099: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764936276.351922      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764936276.396129      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [5]:
# Load dataset
dataset_restaurant = load_dataset("jakartaresearch/semeval-absa", name='restaurant')
train_ds = dataset_restaurant["train"]
test_ds = dataset_restaurant["validation"]

Generating train split:   0%|          | 0/3044 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

In [6]:
# Label map
label_map = {"O": 0, "B": 1, "I": 2}
id2label = {v: k for k, v in label_map.items()}

In [7]:
#load bert model
model = DistilBertForTokenClassification.from_pretrained("distilbert-base-uncased", num_labels=len(label_map))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
#Load bert tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased",
    use_fast=True   # Force Fast tokenizer
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
# Apply PEFT with LoRA
lora_config = LoraConfig(
    r=8,  # Rank of the LoRA matrix
    lora_alpha=16,  # Scaling factor
    lora_dropout=0.1,
    target_modules = ["q_lin", "k_lin", "v_lin"],
    bias="none",
    task_type=TaskType.TOKEN_CLS  # Assuming token classification task
)
peft_model = get_peft_model(model, lora_config)
peft_model.train()

PeftModelForTokenClassification(
  (base_model): LoraModel(
    (model): DistilBertForTokenClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out

In [10]:
# Tokenize
def tokenize_and_align(ex):
    text = ex["text"]
    terms = ex["aspects"]["term"]
    from_idx = ex["aspects"]["from"]
    to_idx = ex["aspects"]["to"]

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    labels = ["O"] * len(encoding["offset_mapping"][0])

    for start, end in zip(from_idx, to_idx):
        span_token_indices = []

        # collect all token positions belonging to this aspect span
        for i, (s, e) in enumerate(encoding["offset_mapping"][0]):
            if s == 0 and e == 0:   # padding tokens
                continue
            if s >= start and e <= end:
                span_token_indices.append(i)

        # assign BIO tags
        if len(span_token_indices) > 0:
            labels[span_token_indices[0]] = "B"       # first token
            for idx in span_token_indices[1:]:        # remaining tokens
                labels[idx] = "I"

    label_ids = [label_map[l] for l in labels]

    return (
        encoding["input_ids"].squeeze(),
        encoding["attention_mask"].squeeze(),
        torch.tensor(label_ids)
    )


In [11]:
# Build tensors
def prepare(ds):
    input_ids, attention_masks, label_ids = [], [], []
    for ex in ds:
        a, b, c = tokenize_and_align(ex)
        input_ids.append(a)
        attention_masks.append(b)
        label_ids.append(c)
    return TensorDataset(torch.stack(input_ids), torch.stack(attention_masks), torch.stack(label_ids))



In [12]:
train_dataset = prepare(train_ds)
test_dataset = prepare(test_ds)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=5e-5)

In [14]:
# Training loop
for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = batch
        outputs = peft_model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch + 1} loss: {loss.item()}")

Epoch 1 loss: 0.004343560431152582
Epoch 2 loss: 0.009631659835577011
Epoch 3 loss: 0.0019844421185553074


In [15]:
# Test data
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)
peft_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = batch
        logits = peft_model(input_ids, attention_mask=attention_mask).logits
        preds = torch.argmax(logits, dim=2)

        preds = preds.flatten()
        labels = labels.flatten()

        all_preds.extend([id2label[p.item()] for p in preds])
        all_labels.extend([id2label[l.item()] for l in labels])

In [16]:
aspects = []
current = []

for token, label in zip(all_preds, all_labels):
    if label == "B":
        if current:
            aspects.append(" ".join(current))
        current = [token]
    elif label == "I" and current:
        current.append(token)
    elif label == "O" and current:
        aspects.append(" ".join(current))
        current = []

if current:
    aspects.append(" ".join(current))

print("Extracted Aspects:", aspects)
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Classification Report:\n", classification_report(all_labels, all_preds, digits=4))

Extracted Aspects: ['B', 'B O', 'B', 'B', 'I I', 'B', 'O', 'B B', 'B I I', 'I I', 'I I', 'O', 'B', 'I I I I', 'B', 'I I I', 'B', 'B', 'B B', 'B', 'B B', 'B', 'B B', 'B', 'I I', 'I', 'B', 'B', 'B', 'B O O B', 'B O O', 'B O O O', 'B', 'B I', 'B I', 'B I O O O', 'B', 'B', 'I I I', 'O O I I', 'B', 'B', 'B', 'B B', 'B B B', 'B', 'B', 'B', 'B', 'I I I', 'I I I', 'B', 'B', 'B', 'B', 'I I', 'B', 'I I I I', 'B B', 'I I I O I I I', 'B', 'O O I', 'B', 'B', 'B O', 'I I I I I I I I I I I I I', 'I I I I', 'O I O', 'I I I', 'I I I I I', 'I I I I I', 'B', 'B O I', 'B', 'I I I I I', 'I I I I', 'B', 'B', 'B', 'B', 'B', 'B', 'B I', 'B', 'B', 'B', 'I I I I', 'B', 'B', 'O', 'B', 'B', 'B', 'I I I', 'B', 'B I I I', 'I I', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'O', 'B', 'I I I', 'B I I', 'B', 'B', 'B', 'B', 'B', 'I I I I I', 'B', 'B', 'B', 'B', 'B', 'B I I', 'O', 'B I', 'B', 'B B B', 'B I', 'B', 'O O I', 'B', 'B', 'B I', 'B I', 'B O O B', 'O', 'B', 'B', 'B', 'I I I', 'I I', 'B B', 'B', 'B', 'B', 'B', 'B', 'I', '